# Supplementary Figure: Receptive Fields and Retinotopy under Single-Depth Protocol

This notebook generates the supplementary figure characterizing receptive fields (RFs) and spatial retinotopy measured under the single-depth visual stimulation protocol (`SpheresPermTubeReward`).

### Panels:
- **A**: Model formulation schematic ($\Delta F/F_0 = S\mathbf{b}$)
- **B**: Reconstructed visual stimulus frames across 8 depths (5 to 640 cm)
- **C**: Depth tuning curves and depth-resolved RF maps for 3 example V1 neurons (ROIs 319, 720, 120)
- **D**: 3D spatio-temporal RF profiles in visual space (Elevation $\times$ Azimuth $\times$ Virtual depth)
- **E**: Distribution of the proportion of neurons with significant RFs across recording sessions
- **F, G, H**: Spatial retinotopic organization of an example V1 FOV:
  - **F**: Preferred Azimuth (degrees)
  - **G**: Preferred Elevation (degrees)
  - **H**: Preferred Virtual Depth (cm) with anatomical 2P mean image inset


In [ ]:
%reload_ext autoreload
%autoreload 2

import pickle
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

import flexiznam as flz
from cottage_analysis.analysis import spheres, common_utils, roi_location
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting import rf_plots, depth_selectivity_plots, plotting_utils
from cottage_analysis.summary_analysis import get_session_list
from cottage_analysis.plotting import style

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

In [ ]:
project = "hey2_3d-vision_foodres_20220101"
flexilims_session = flz.get_flexilims_session(project)
from v1_depth_map.paths import get_figures_roots

READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
recompute_summary = False  # Set True to recompute and re-cache population RF summaries

## Load Data
### 1. Example Session Data


In [ ]:
# Load data for example session and neurons
session_name_rf = "PZAH8.2h_S20230116"

vs_df_rf, trials_df_rf = spheres.sync_all_recordings(
    session_name=session_name_rf,
    flexilims_session=flexilims_session,
    filter_datasets={"anatomical_only": 3, "ast_neuropil": False},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward",
    photodiode_protocol=5,
    return_volumes=True,
)

frames_rf, imaging_df_rf = spheres.regenerate_frames_all_recordings(
    session_name=session_name_rf,
    flexilims_session=flexilims_session,
    filter_datasets={"anatomical_only": 3, "ast_neuropil": False},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward",
    photodiode_protocol=5,
    return_volumes=True,
    resolution=1,
)

neurons_ds_rf = pipeline_utils.create_neurons_ds(
    session_name=session_name_rf,
    flexilims_session=flexilims_session,
    project=project,
    conflicts="skip",
)
neurons_df_rf = pd.read_pickle(neurons_ds_rf.path_full)

# Load suite2p ROI metadata for FOV spatial plots
suite2p_ds = flz.get_datasets(
    flexilims_session=flexilims_session,
    origin_name=session_name_rf,
    dataset_type="suite2p_rois",
    filter_datasets={"anatomical_only": 3},
    allow_multiple=False,
    return_dataseries=False,
)
iscell = np.load(suite2p_ds.path_full / "plane0" / "iscell.npy", allow_pickle=True)[:, 0]
neurons_df_rf["iscell"] = iscell
stat = np.load(suite2p_ds.path_full / "plane0" / "stat.npy", allow_pickle=True)
ops = np.load(suite2p_ds.path_full / "plane0" / "ops.npy", allow_pickle=True).item()

### 2. Population RF Summary Data


In [ ]:
# Load summary data across all recording sessions
if recompute_summary:
    all_sig, all_sig_ipsi, neurons_df_all = rf_plots.load_sig_rf(
        flexilims_session=flexilims_session,
        session_list=get_session_list.get_sessions(
            flexilims_session=flexilims_session,
            exclude_openloop=False,
            exclude_pure_closedloop=False,
            v1_only=True,
        ),
        n_std=6,
    )
    neurons_df_all = roi_location.align_across_mice(neurons_df_all)
    neurons_df_all.to_pickle(SAVE_ROOT / "rf_neurons_df_all.pkl")

    with open(SAVE_ROOT / "rf_all_sig.pkl", "wb") as f:
        pickle.dump(all_sig, f, protocol=pickle.HIGHEST_PROTOCOL)
    with open(SAVE_ROOT / "rf_all_sig_ipsi.pkl", "wb") as f:
        pickle.dump(all_sig_ipsi, f, protocol=pickle.HIGHEST_PROTOCOL)
else:
    neurons_df_all = pd.read_pickle(READ_ROOT / "rf_neurons_df_all.pkl")
    with open(READ_ROOT / "rf_all_sig.pkl", "rb") as f:
        all_sig = pickle.load(f)
    with open(READ_ROOT / "rf_all_sig_ipsi.pkl", "rb") as f:
        all_sig_ipsi = pickle.load(f)

common_utils.add_one_sided_spearman_significance(
    neurons_df_all,
    rval_col="depth_tuning_test_spearmanr_rval_closedloop",
    pval_col="depth_tuning_test_spearmanr_pval_closedloop",
    out_col="is_depth_neuron",
)
depth_selective = (neurons_df_all["iscell"] == 1) & (neurons_df_all["is_depth_neuron"])
rf_sig = neurons_df_all["rf_sig"] & depth_selective
print(f"{np.sum(rf_sig)} RF sig neurons out of {np.sum(depth_selective)} depth selective neurons ({np.sum(rf_sig)/np.sum(depth_selective):.1%})")

## Panels B & C: Reconstructed Stimuli & Example Neuron RFs


In [ ]:
fontsize_dict = FONTSIZE_DICT
fig = plt.figure(figsize=(18 / 2.54, 18 / 2.54))

# Panel B: Example reconstructed stimuli frame
trial_idx = 4
example_trial = trials_df_rf.iloc[trial_idx]
depths = np.sort(trials_df_rf.depth.unique())
idepth = np.nonzero(depths == example_trial.depth)[0][0]
iframe = (trials_df_rf.iloc[trial_idx].imaging_stim_start + 10,)
rf_plots.plot_stimulus_frame(
    frame=frames_rf[iframe, :, frames_rf.shape[2] // 2 :].squeeze(),
    idepth=idepth,
    depths=depths,
    position=[0, 0.74, 0.3, 0.4],
    fontsize_dict=fontsize_dict,
    plot_prop=0.9,
)

# Panel C: Example receptive field of 3 V1 neurons
SELECT_ROIS = [
    319,
    720,
    120,
]
for iroi, (roi, linecolor) in enumerate(zip(SELECT_ROIS, ["red", "green", "blue"])):
    fig.add_axes([0.267 + 0.165 * iroi, 0.9, 0.05, 0.05])
    depth_tuning_kwargs = dict(
        rs_thr=None,
        plot_fit=True,
        plot_smooth=False,
        linewidth=1,
        closed_loop=1,
        fontsize_dict=fontsize_dict,
        markersize=5,
        markeredgecolor='w',
    )
    depth_selectivity_plots.plot_depth_tuning_curve(
        neurons_df=neurons_df_rf,
        trials_df=trials_df_rf,
        roi=roi,
        linecolor=linecolor,
        ylim_precision_base=5,
        ylim_precision=2,
        **depth_tuning_kwargs,
    )
    plt.ylabel("")
    plt.xlabel("")
    plt.gca().set_xticklabels([])
    plt.gca().tick_params(length=1.5)
    if iroi == len(SELECT_ROIS) // 2:
        xlabel = "Azimuth (degrees)"
    else:
        xlabel = ""
    if iroi == 0:
        ylabel = "Elevation (degrees)"
    else:
        ylabel = ""
    rf_plots.plot_rf(
        neurons_df=neurons_df_rf,
        roi=roi,
        ndepths=8,
        frame_shape=(16, 24),
        position=[0.055 + 0.165 * iroi, 0.828, 0.5, 0.5],
        plot_prop=0.9,
        xlabel=xlabel,
        ylabel=ylabel,
        fontsize_dict=fontsize_dict,
    )

style.savefig(
    SAVE_ROOT / "figsupp_single_depth_rf_examples.svg", fig=fig, bbox_inches="tight"
)
plt.show()

## Panel D: 3D Spatio-Temporal RF Profiles


In [ ]:
# Plot 3D spatio-temporal receptive fields in visual space
depths = np.sort(trials_df_rf.depth.unique())
rf_plots.plot_rf_3d(
    neurons_df_rf,
    SELECT_ROIS,
    depths,
    SAVE_ROOT / "figsupp_single_depth_3d_rfs.pdf",
    dict(label=10, tick=10),
)

## Panels E, F, G, H: Population Yield & Spatial Retinotopy in V1 FOV


In [ ]:
depths = np.sort(trials_df_rf.depth.unique())
fontsize_dict = FONTSIZE_DICT
fig = plt.figure(figsize=(18 / 2.54, 18 / 2.54))

# Panel E: Proportion of significant RFs histogram across sessions
fig.add_axes([0.1, 0.2, 0.2, 0.2])
rf_plots.plot_sig_rf_perc(
    all_sig=all_sig,
    all_sig_ipsi=all_sig_ipsi,
    plot_type="hist",
    hist_color="cornflowerblue",
    hist_edgecolor="royalblue",
    bins=np.arange(0, 1, 0.1),
    fontsize_dict=fontsize_dict,
)
print(f"Median ipsilateral significant RF proportion: {np.median(all_sig_ipsi):.2%}")
print(f"Median contralateral significant RF proportion: {np.median(all_sig):.2%}")

# Panel F: Azimuth map across example FOV
fig.add_axes([0.07, 0.5, 0.25, 0.25])
im_azi = depth_selectivity_plots.plot_example_fov(
    neurons_df=neurons_df_rf,
    ops=ops,
    stat=stat,
    ndepths=len(depths),
    col="rf_azi",
    cmap=cm.YlOrRd.reversed(),
    background_color=np.array([0, 0, 0]),
    fontsize_dict=fontsize_dict,
    fov_width=661,
)

# Panel G: Elevation map across example FOV
fig.add_axes([0.37, 0.5, 0.25, 0.25])
im_ele = depth_selectivity_plots.plot_example_fov(
    neurons_df=neurons_df_rf,
    ops=ops,
    stat=stat,
    ndepths=len(depths),
    col="rf_ele",
    cmap=cm.YlOrRd.reversed(),
    background_color=np.array([0, 0, 0]),
    fontsize_dict=fontsize_dict,
    fov_width=661,
)

# Panel H: Preferred Depth map across example FOV + anatomical meanImg inset
fig.add_axes([0.65, 0.5, 0.25, 0.25])
im_depth = depth_selectivity_plots.plot_example_fov(
    neurons_df=neurons_df_rf,
    ops=ops,
    stat=stat,
    ndepths=len(depths),
    col="preferred_depth_closedloop",
    cmap=cm.cool.reversed(),
    background_color=np.array([0, 0, 0]),
    fontsize_dict=fontsize_dict,
    fov_width=661,
)
plotting_utils.plot_white_rectangle(0.805, 0.685, 0.2, 0.2)
fig.add_axes([0.81, 0.68, 0.1, 0.1])
depth_selectivity_plots.plot_fov_mean_img(ops["meanImg"], fov_width=661)

style.savefig(
    SAVE_ROOT / "figsupp_single_depth_rf_fov_and_yield.svg", fig=fig, dpi=300
)
plt.show()